In [7]:
import sys
import os

# go two levels up to reach project root then src
sys.path.append(os.path.abspath("../../src"))

print("Path added:", os.path.abspath("../../src"))


Path added: c:\Users\Anandita\Desktop\Uploads\augenblick\round 2\codebase\AUGENBLICK-HACKATHON-2026\src


In [8]:
corpus_path = "data/devanagari_corpus.txt"

In [9]:
sindhi = "आयो लाल, सभई चायो, झूलेलाल!"
marathi = "गणपती बप्पा मोरया, पुढच्या वर्षी लवकर या!"

In [10]:
def fertility(tokenizer, text):
    encoding = tokenizer.encode(text)
    tokens = encoding.tokens
    words = text.split()
    return len(tokens) / len(words), tokens

In [19]:
corpus_path = "data/devanagari_corpus.txt"
small_path = "data/dev_small.txt"

lines = []

with open(corpus_path, encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 40000:   # keep first 40k lines
            break
        lines.append(line)

with open(small_path, "w", encoding="utf-8") as f:
    f.writelines(lines)

print("Small corpus created:", small_path)
print("Total lines:", len(lines))

Small corpus created: data/dev_small.txt
Total lines: 40000


In [20]:
from abctokz.tokenizer import AugenblickTokenizer
from abctokz.config.schemas import TokenizerConfig

vocab_sizes = [100, 400, 800]
models = ["bpe", "unigram"]

results = []

corpus = "data/dev_small.txt"

for model_type in models:
    for vocab_size in vocab_sizes:

        print("Training:", model_type, "vocab:", vocab_size)

        config_dict = {
            "model": {"type": model_type},
            "trainer": {"type": model_type, "vocab_size": vocab_size},
            "normalizer": {"type": "devanagari"},
            "pretokenizer": {"type": "whitespace"},
            "add_bos": False,
            "add_eos": False
        }

        config = TokenizerConfig(**config_dict)

        tokenizer = AugenblickTokenizer.from_config(config)

        tokenizer.train([corpus], config)

        # compute fertility
        f_sindhi, tok_s = fertility(tokenizer, sindhi)
        f_marathi, tok_m = fertility(tokenizer, marathi)

        results.append({
            "model": model_type,
            "vocab_size": vocab_size,
            "sindhi_fertility": f_sindhi,
            "marathi_fertility": f_marathi
        })

        print("Sindhi tokens:", tok_s)
        print("Marathi tokens:", tok_m)
        print()

print(results)

Training: bpe vocab: 100
Sindhi tokens: ['आ', '##य', '##ो', 'ल', '##ा', '##ल', '##,', 'स', '##भ', '##ई', 'च', '##ा', '##य', '##ो', '##,', 'झ', '##ू', '##ल', '##े', '##ल', '##ा', '##ल', '##!']
Marathi tokens: ['ग', '##ण', '##प', '##त', '##ी', 'ब', '##प', '##्', '##प', '##ा', 'म', '##ो', '##र', '##य', '##ा', '##,', 'प', '##ु', '##ढ', '##च', '##्', '##य', '##ा', 'व', '##र', '##्', '##ष', '##ी', 'ल', '##व', '##क', '##र', 'य', '##ा', '##!']

Training: bpe vocab: 400
Sindhi tokens: ['आ', '##य', '##ो', 'ल', '##ाल', '##,', 'स', '##भ', '##ई', 'च', '##ा', '##य', '##ो', '##,', 'झ', '##ू', '##ले', '##ल', '##ाल', '##!']
Marathi tokens: ['ग', '##ण', '##प', '##ती', 'ब', '##प', '##्', '##प', '##ा', 'म', '##ो', '##र', '##या', '##,', 'प', '##ु', '##ढ', '##च', '##्', '##या', 'व', '##र्', '##ष', '##ी', 'ल', '##व', '##कर', 'य', '##ा', '##!']

Training: bpe vocab: 800
Sindhi tokens: ['आ', '##य', '##ो', 'ल', '##ाल', '##,', 'स', '##भ', '##ई', 'च', '##ाय', '##ो', '##,', 'झ', '##ू', '##ले', '##ल', '##ाल', '##!'

In [18]:
with open(corpus_path, encoding="utf-8") as f:
    print(sum(1 for _ in f))

487246


In [ ]:
#Experiment 10 - trying different configurations 

In [3]:
import sys
import os

# make repo package importable
sys.path.append(os.path.abspath("../../src"))

from abctokz.tokenizer import AugenblickTokenizer
from abctokz.config.schemas import TokenizerConfig

print("Imports successful")

Imports successful


In [1]:
configs = [
    {"model": "bpe", "vocab": 100, "pretokenizer": "whitespace"},
    {"model": "bpe", "vocab": 800, "pretokenizer": "whitespace"},
    {"model": "bpe", "vocab": 400, "pretokenizer": "punctuation"},
    {"model": "unigram", "vocab": 400, "pretokenizer": "whitespace"}
]

In [11]:
tradeoff_results = []

for cfg in configs:

    print("Running:", cfg)

    config_dict = {
        "model": {"type": cfg["model"]},
        "trainer": {"type": cfg["model"], "vocab_size": cfg["vocab"]},
        "normalizer": {"type": "devanagari"},
        "pretokenizer": {"type": cfg["pretokenizer"]},
        "add_bos": False,
        "add_eos": False
    }

    config = TokenizerConfig(**config_dict)

    tokenizer = AugenblickTokenizer.from_config(config)

    tokenizer.train(["data/dev_small.txt"], config)

    f_sindhi, tok_s = fertility(tokenizer, sindhi)
    f_marathi, tok_m = fertility(tokenizer, marathi)

    tradeoff_results.append({
        "model": cfg["model"],
        "vocab": cfg["vocab"],
        "pretokenizer": cfg["pretokenizer"],
        "sindhi_fertility": f_sindhi,
        "marathi_fertility": f_marathi
    })

print(tradeoff_results)

Running: {'model': 'bpe', 'vocab': 100, 'pretokenizer': 'whitespace'}
Running: {'model': 'bpe', 'vocab': 800, 'pretokenizer': 'whitespace'}
Running: {'model': 'bpe', 'vocab': 400, 'pretokenizer': 'punctuation'}
Running: {'model': 'unigram', 'vocab': 400, 'pretokenizer': 'whitespace'}
[{'model': 'bpe', 'vocab': 100, 'pretokenizer': 'whitespace', 'sindhi_fertility': 4.6, 'marathi_fertility': 5.0}, {'model': 'bpe', 'vocab': 800, 'pretokenizer': 'whitespace', 'sindhi_fertility': 3.8, 'marathi_fertility': 3.7142857142857144}, {'model': 'bpe', 'vocab': 400, 'pretokenizer': 'punctuation', 'sindhi_fertility': 3.8, 'marathi_fertility': 4.142857142857143}, {'model': 'unigram', 'vocab': 400, 'pretokenizer': 'whitespace', 'sindhi_fertility': 4.6, 'marathi_fertility': 4.428571428571429}]


In [12]:
import pandas as pd

df_tradeoff = pd.DataFrame(tradeoff_results)

df_tradeoff

,model,vocab,pretokenizer,sindhi_fertility,marathi_fertility
0,bpe,100,whitespace,4.6,5.000000
1,bpe,800,whitespace,3.8,3.714286
2,bpe,400,punctuation,3.8,4.142857
3,unigram,400,whitespace,4.6,4.428571
